In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

from sklearn.metrics.pairwise import cosine_similarity

### Loading Animelist

In [2]:
df_AnimeList = pd.DataFrame()
for f in list(Path('../Scraping/Season').rglob('*.csv')):
    if re.search(r'.ipynb_checkpoints', str(f)) is not None:
        continue

    df_Read = pd.read_csv(f)
    df_AnimeList = pd.concat([df_AnimeList, df_Read])

del df_Read

In [3]:
df_AnimeList['MAL_id'] = df_AnimeList['Link'].apply(lambda x : int(re.findall(r'anime/(\d+)',x)[0]))
df_AnimeList = df_AnimeList.drop(columns=['Link', 'Unnamed: 0'])
df_AnimeList = df_AnimeList.set_index('MAL_id')
df_AnimeList = df_AnimeList.drop_duplicates()
df_AnimeList

,Title
MAL_id,
30,Shinseiki Evangelion
852,Gokinjo Monogatari
495,Bakuretsu Hunters
1567,Kaitou Saint Tail
116,Shinpi no Sekai El-Hazard (TV)
...,...
54145,Cardfight!! Vanguard: D Series Season 9
53722,Bullet/Bullet
60726,My Melody & Kuromi


### Loading User Ratings

In [4]:
df_Raw = pd.DataFrame()
for f in list(Path('../Scraping/Users/Watchlist').rglob('*.csv')):
    if re.search(r'.ipynb_checkpoints', str(f)) is not None:
        continue

    df_Read = pd.read_csv(f)
    df_Read['User'] = f.name.replace('.csv', '')
    df_Raw = pd.concat([df_Raw, df_Read])

del df_Read

In [5]:
df_Raw = df_Raw.drop(columns=['Unnamed: 0'])
df_Raw = df_Raw.dropna()

df_Raw['MAL_id'] = df_Raw['MAL_id'].astype(int)
df_Raw

,MAL_id,Title,Score,Type,User
4,40028,Shingeki_no_Kyojin__The_Final_Season,10.0,TV,-Norza-
20,40938,Hige_wo_Soru_Soshite_Joshikousei_wo_Hirou,9.0,TV,-Norza-
32,1735,Naruto__Shippuuden,9.0,TV,-Norza-
59,3455,To_LOVE-Ru,7.0,TV,-Norza-
61,16011,Tokyo_Ravens,8.0,TV,-Norza-
...,...,...,...,...,...
223,36889,Oyasumi_Sex,7.0,OVA,_AverageWatcher_
224,49987,Papa_Katsu,6.0,OVA,_AverageWatcher_
226,59120,Shounen_ga_Otona_ni_Natta_Natsu,3.0,OVA,_AverageWatcher_
227,49329,Soshite_Watashi_wa_Sensei_ni,4.0,OVA,_AverageWatcher_


### User-Anime Score Matrix

In [6]:
matrix_Raw = df_Raw.pivot_table(index='User', columns='MAL_id', values='Score', aggfunc='mean')
matrix_Raw = matrix_Raw.fillna(0)
matrix_Raw = matrix_Raw.astype(np.int8)
matrix_Raw

MAL_id,1,5,6,7,8,15,16,17,18,19,...,60108,60182,60407,60410,60443,60512,60544,60553,60629,60736
User,,,,,,,,,,,,,,,,,,,,,
-Norza-,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0neiros,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24xnyk,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2alvin,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ACRONYM117,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wellfuckthat,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
xLucqd,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
xquasarx,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### User-User Similarity Matrix

In [7]:
matrix_Users = pd.DataFrame(cosine_similarity(matrix_Raw), index=matrix_Raw.index, columns=matrix_Raw.index)
matrix_Users.head(10)

User,-Norza-,0neiros,24xnyk,2alvin,ACRONYM117,Actual_Hana,Aelter,Ahegaofan94,AlphaSchnecke,Arcanon,...,tazuosa,themastergooner,traed,tytl,valoon,wellfuckthat,xLucqd,xquasarx,yanzz22,zhorahh
User,,,,,,,,,,,,,,,,,,,,,
-Norza-,1.000000,0.162018,0.069213,0.00000,0.316455,0.096524,0.000000,0.000000,0.321150,0.207100,...,0.000000,0.000000,0.039509,0.000000,0.000000,0.109467,0.144350,0.018825,0.000000,0.000000
0neiros,0.162018,1.000000,0.072180,0.00000,0.218530,0.140174,0.008516,0.000000,0.142978,0.270482,...,0.021637,0.000000,0.121981,0.000000,0.004780,0.203071,0.110429,0.072594,0.000000,0.097975
24xnyk,0.069213,0.072180,1.000000,0.09202,0.044967,0.012534,0.073899,0.219096,0.047476,0.036936,...,0.000000,0.055743,0.024401,0.183349,0.013401,0.000000,0.000000,0.000000,0.248038,0.000000
2alvin,0.000000,0.000000,0.092020,1.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.120290,0.000000,0.000000,0.000000,0.000000,0.045529,0.000000
ACRONYM117,0.316455,0.218530,0.044967,0.00000,1.000000,0.239364,0.000000,0.000000,0.258456,0.218094,...,0.071191,0.000000,0.106998,0.000000,0.000000,0.143739,0.195954,0.148539,0.000000,0.138530
Actual_Hana,0.096524,0.140174,0.012534,0.00000,0.239364,1.000000,0.000000,0.000000,0.167723,0.128019,...,0.084276,0.000000,0.131124,0.000000,0.025087,0.050346,0.079326,0.157304,0.000000,0.088072
Aelter,0.000000,0.008516,0.073899,0.00000,0.000000,0.000000,1.000000,0.164688,0.010821,0.009035,...,0.000000,0.079863,0.000000,0.132643,0.000000,0.000000,0.000000,0.000000,0.088595,0.000000
Ahegaofan94,0.000000,0.000000,0.219096,0.00000,0.000000,0.000000,0.164688,1.000000,0.008135,0.000000,...,0.000000,0.000000,0.000000,0.328987,0.006549,0.000000,0.000000,0.000000,0.183229,0.000000
AlphaSchnecke,0.321150,0.142978,0.047476,0.00000,0.258456,0.167723,0.010821,0.008135,1.000000,0.166593,...,0.000000,0.088067,0.099379,0.000000,0.008677,0.072163,0.205767,0.056346,0.003203,0.092196


## Test User Actual_Hana
Gather Top 10 Most Similar Users

In [8]:
USER = 'Actual_Hana'

similar_users = matrix_Users.loc[USER].sort_values(ascending=False)[1:11]
similar_users

User
Deadpulse         0.288036
manakamukaido     0.269933
saxzer            0.259764
lYukki            0.257342
BELiCARiOs        0.256026
ACRONYM117        0.239364
NachoPizza        0.239337
Gabby_NEW         0.238397
SupaJanoskiiii    0.232487
silvxr            0.228730
Name: Actual_Hana, dtype: float64

## Shows User Watched. Others have Watched Some

In [9]:
alreadyWatched = matrix_Raw.loc[USER] > 0
matrix_Raw.loc[np.append(USER, similar_users.index), alreadyWatched]

MAL_id,20,22,121,123,226,270,355,356,384,853,...,22145,22199,22319,22535,23273,23283,23317,23673,27899,28223
User,,,,,,,,,,,,,,,,,,,,,
Actual_Hana,10,9,10,10,9,9,9,9,10,9,...,8,9,10,10,10,10,8,10,10,10
Deadpulse,0,0,0,0,0,0,0,0,0,0,...,0,0,10,0,8,0,0,0,6,0
manakamukaido,0,9,0,0,0,0,8,0,0,0,...,0,0,0,0,0,0,0,0,0,0
saxzer,9,0,0,9,0,0,0,10,0,0,...,9,0,7,0,7,0,9,7,5,10
lYukki,0,0,0,0,0,0,0,0,0,9,...,0,0,0,0,0,0,0,0,0,0
BELiCARiOs,0,0,0,0,0,0,0,0,0,9,...,0,0,0,8,10,0,0,9,0,8
ACRONYM117,0,0,0,0,0,0,9,0,0,0,...,0,8,0,8,7,0,0,0,0,8
NachoPizza,0,0,0,0,0,0,9,0,0,0,...,0,8,10,9,10,0,0,0,8,0
Gabby_NEW,0,0,0,0,0,0,0,0,0,0,...,0,9,10,10,8,9,0,0,8,0


## Shows User hasn't Watched. Others have

In [10]:
matrix_Other_User_Ratings = matrix_Raw.loc[similar_users.index, ~alreadyWatched]
matrix_Other_User_Ratings

MAL_id,1,5,6,7,8,15,16,17,18,19,...,60108,60182,60407,60410,60443,60512,60544,60553,60629,60736
User,,,,,,,,,,,,,,,,,,,,,
Deadpulse,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
manakamukaido,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
saxzer,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
lYukki,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
BELiCARiOs,0,0,5,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ACRONYM117,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NachoPizza,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Gabby_NEW,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SupaJanoskiiii,0,0,0,0,0,0,0,0,0,10,...,0,0,0,0,0,0,0,0,0,0


### Recommendations' Avg Score by Similar Users

In [11]:
reccomended_Shows = matrix_Other_User_Ratings.mean(axis=0).sort_values(ascending=False)[:10]
df_Reccomendations = df_AnimeList.loc[reccomended_Shows.index].copy()
df_Reccomendations['Avg_Score'] = reccomended_Shows.values
df_Reccomendations

,Title,Avg_Score
MAL_id,,
24833,Ansatsu Kyoushitsu,6.2
9989,Ano Hi Mita Hana no Namae wo Bokutachi wa Mada...,6.0
13759,Sakura-sou no Pet na Kanojo,5.8
2904,Code Geass: Hangyaku no Lelouch R2,5.7
12189,Hyouka,5.3
14741,Chuunibyou demo Koi ga Shitai!,4.7
17895,Golden Time,4.6
14813,Yahari Ore no Seishun Love Comedy wa Machigatt...,4.6
16067,Nagi no Asu kara,4.5


In [21]:
def Recommend(USER):
    similar_users = matrix_Users.loc[USER].sort_values(ascending=False)[1:11]
    alreadyWatched = matrix_Raw.loc[USER] > 0
    matrix_Other_User_Ratings = matrix_Raw.loc[similar_users.index, ~alreadyWatched]

    reccomended_Shows = matrix_Other_User_Ratings.mean(axis=0).sort_values(ascending=False)[:10]
    df_Reccomendations = df_AnimeList.loc[reccomended_Shows.index].copy()
    df_Reccomendations['Avg_Recommenders_Score'] = reccomended_Shows.values
    df_Reccomendations = df_Reccomendations.rename(columns={'Title': "Others' Top 10"})
    return df_Reccomendations

def GetPersonalTop10(USER):
    USER_top10 = matrix_Raw.loc[USER].sort_values(ascending=False)[:10]
    df = df_AnimeList.loc[USER_top10.index].copy()
    df['Personal_Score'] = USER_top10.values
    df = df.rename(columns={'Title': f"{USER}'s Top 10"})
    return df

## Test Mainstream

In [23]:
USER = 'Chiirou'
pd.concat([GetPersonalTop10(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,Chiirou's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,28497,Rokka no Yuusha,10,13759,Sakura-sou no Pet na Kanojo,8.3
1,14741,Chuunibyou demo Koi ga Shitai!,10,18897,Nisekoi,7.4
2,27989,Hibike! Euphonium,10,30276,One Punch Man,7.2
3,31376,Flying Witch,10,10793,Guilty Crown,6.5
4,2026,Hayate no Gotoku!,9,27775,Plastic Memories,6.1
5,29803,Overlord,9,25777,Shingeki no Kyojin Season 2,6.1
6,26441,Junketsu no Maria,9,35507,Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu e,6.0
7,28121,Dungeon ni Deai wo Motomeru no wa Machigatteir...,9,14813,Yahari Ore no Seishun Love Comedy wa Machigatt...,5.9
8,31564,Sansha Sanyou,9,17895,Golden Time,5.8
9,4081,Natsume Yuujinchou,9,21995,Ao Haru Ride,5.8


## Test Gooner

In [22]:
USER = 'omanko1'
pd.concat([GetPersonalTop10(USER).reset_index(), Recommend(USER).reset_index()], axis=1)

,MAL_id,omanko1's Top 10,Personal_Score,MAL_id,Others' Top 10,Avg_Recommenders_Score
0,49295,Chizuru-chan Kaihatsu Nikki,10,11827,Kuroinu: Kedakaki Seijo wa Hakudaku ni Somaru,7.3
1,59119,Imaria,10,38470,Saimin Seishidou,6.7
2,50192,Akane wa Tsumare Somerareru,10,24641,Baku Ane: Otouto Shibocchau zo! The Animation,6.3
3,55146,Kunoichi Ryoujokuden Ajisai,9,36720,Youkoso! Sukebe Elf no Mori e,6.3
4,2539,Itazura The Animation,9,39042,Gaki ni Modotte Yarinaoshi!!!,6.0
5,35581,Oide yo! Mizuryuu Kei Land,9,48698,Mako-chan Kaihatsu Nikki,5.6
6,32587,Shoujo Ramune,9,33322,JK Bitch ni Shiboraretai,5.5
7,39530,Ecchi na Oneechan ni Shiboraretai,9,38866,Seikatsu Shuukan The Animation,5.4
8,31886,Dropout,9,29575,Mankitsu Happening,5.4
9,43466,Himawari wa Yoru ni Saku,9,21829,Fella Pure: Mitarashi-san Chi no Jijou The Ani...,5.4
